# Match an image pair with UltraMatch

Run from the repository root with the UltraMatch checkpoint in `weights/outdoor.ckpt`. Both examples use the same weights. The matcher returns corresponding pixel coordinates and one confidence score per match.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch

from src.config.default import get_cfg_defaults
from src.ultramatch import UltraMatch
from src.utils.dataset import read_megadepth_gray, read_scannet_gray
from src.utils.misc import lower_config
from src.utils.plotting import make_matching_figure

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = "weights/outdoor.ckpt"


def load_matcher(model_config, data_config):
    config = get_cfg_defaults()
    config.merge_from_file(model_config)
    config.merge_from_file(data_config)

    matcher = UltraMatch(lower_config(config.ULTRAMATCH))
    matcher.load_state_dict(torch.load(checkpoint, map_location="cpu")["state_dict"])
    matcher.to(device).eval()
    if config.ULTRAMATCH.REPARAMETERIZE:
        matcher.reparameterize_for_inference()
    if config.ULTRAMATCH.FUSE_ATTENTION_PROJECTIONS:
        matcher.fuse_attention_for_inference()
    return matcher, config


@torch.inference_mode()
def match_pair(matcher, batch):
    batch = {key: value.unsqueeze(0).to(device) for key, value in batch.items()}
    matcher(batch)
    return tuple(batch[key].cpu().numpy() for key in ("mkpts0_f", "mkpts1_f", "mconf"))


def draw_matches(images, points0, points1, confidence):
    selected = confidence.argsort()[::-1][:200]
    make_matching_figure(
        *images, points0[selected], points1[selected], plt.cm.viridis(confidence[selected]),
        text=["UltraMatch", f"{len(confidence)} matches ({len(selected)} shown)"],
    )
    plt.show()

## Indoor pair

Resize the ScanNet images to the indoor evaluation resolution and draw matches in that coordinate system.

In [ ]:
matcher, config = load_matcher(
    "configs/ultramatch/indoor/ultramatch_indoor.py", "configs/data/scannet_test_1500.py"
)
folder = Path("assets/scannet_sample_images")
paths = [folder / "scene0707_00_15.jpg", folder / "scene0707_00_45.jpg"]
size = (config.ULTRAMATCH.TEST_RES_W, config.ULTRAMATCH.TEST_RES_H)
batch = {f"image{i}": read_scannet_gray(str(path), size) for i, path in enumerate(paths)}
points0, points1, confidence = match_pair(matcher, batch)
images = [cv2.resize(cv2.imread(str(path)), size)[:, :, ::-1] for path in paths]
draw_matches(images, points0, points1, confidence)

## Outdoor pair

The outdoor evaluation loader resizes each image by its long edge and pads it to a square. The masks exclude padding and the scale factors map the predicted points back to the original photographs.

In [ ]:
matcher, config = load_matcher(
    "configs/ultramatch/outdoor/ultramatch.py", "configs/data/megadepth_test_1500.py"
)
folder = Path("assets/phototourism_sample_images")
paths = [
    folder / "united_states_capitol_26757027_6717084061.jpg",
    folder / "united_states_capitol_98169888_3347710852.jpg",
]
batch = {}
stride = config.ULTRAMATCH.LOCAL_RESOLUTION
for i, path in enumerate(paths):
    image, mask, scale = read_megadepth_gray(
        str(path), resize=config.ULTRAMATCH.TEST_RES_W, df=config.DATASET.MGDPT_DF, padding=True
    )
    batch[f"image{i}"] = image
    batch[f"mask{i}"] = mask[::stride, ::stride].contiguous()
    batch[f"scale{i}"] = scale

points0, points1, confidence = match_pair(matcher, batch)
images = [cv2.imread(str(path))[:, :, ::-1] for path in paths]
draw_matches(images, points0, points1, confidence)